# MoneyPrinterTurbo — abrir desde Samsung 📱

Pulsa **▶** en la celda de abajo. El programa reutiliza la instalación si está disponible; si no, instala el repositorio `Danifloo/MoneyPrinterTurbo`.

Cuando lo pida, escribe **tu correo en el campo privado de Colab**: Cloudflare enviará allí un código de entrada. No compartas ese código, la URL ni las claves API. El enlace funciona solo mientras Colab esté activo.

Este cuaderno es para pruebas personales. **No publica videos** ni concede acceso permanente. Puede usar cuota gratuita de Colab y servicios de IA con sus propios costos.

In [ ]:
# MoneyPrinterTurbo en Colab: un solo paso, sin ngrok.
import sys, shutil, subprocess, time, re
from pathlib import Path
from getpass import getpass
from urllib.request import urlopen, urlretrieve

REPO = Path("/content/MoneyPrinterTurbo")
REPO_URL = "https://github.com/Danifloo/MoneyPrinterTurbo.git"
PORT = 8501

# Reutiliza la instalacion anterior si sigue disponible.
if not (REPO / "webui" / "Main.py").is_file():
    if REPO.exists():
        raise RuntimeError("La carpeta MoneyPrinterTurbo existe, pero faltan archivos. Revisa Colab antes de continuar.")
    print("Descargando tu proyecto...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

if not (REPO / ".venv" / "bin" / "python").is_file():
    print("Preparando dependencias...")
    subprocess.run(["uv", "python", "install", "3.11"], check=True, cwd=REPO)
    subprocess.run(["uv", "sync", "--frozen", "--python", "3.11"], check=True, cwd=REPO)

# Comprueba si el servidor de la celda anterior sigue vivo.
def healthy():
    try:
        with urlopen(f"http://127.0.0.1:{PORT}/_stcore/health", timeout=3) as response:
            return response.status == 200
    except Exception:
        return False

if not healthy():
    print("Iniciando MoneyPrinterTurbo...")
    server_log = Path("/content/mpt-servidor.log")
    server_log_file = server_log.open("a")
    server_proc = subprocess.Popen(
        ["uv", "run", "streamlit", "run", "webui/Main.py",
         f"--server.port={PORT}", "--server.address=127.0.0.1",
         "--server.headless=true", "--browser.gatherUsageStats=false"],
        cwd=REPO, stdout=server_log_file, stderr=subprocess.STDOUT
    )
    for _ in range(60):
        if healthy():
            break
        if server_proc.poll() is not None:
            raise RuntimeError("El servidor se cerro. Logs:\n" + server_log.read_text(errors="replace")[-2400:])
        time.sleep(2)
    else:
        raise RuntimeError("El servidor no respondio. Logs:\n" + server_log.read_text(errors="replace")[-2400:])

print("Servidor confirmado. Ahora protege el acceso con tu correo.")
mail = getpass("Correo que recibira el codigo de acceso (no se muestra): ").strip()
if not re.fullmatch(r"[^\\s@]+@[^\\s@]+\\.[^\\s@]+", mail):
    raise ValueError("Correo invalido. Ejecuta de nuevo y escribe un correo valido.")

binary = Path("/content/cloudflared")
if not binary.is_file():
    print("Descargando Cloudflare Tunnel...")
    urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        str(binary)
    )
    binary.chmod(0o755)

# Evita dejar corriendo un tunel anterior de esta misma celda.
try:
    if "mpt_cloudflare_proc" in globals() and mpt_cloudflare_proc.poll() is None:
        mpt_cloudflare_proc.terminate()
except Exception:
    pass

tunnel_log = Path("/content/mpt-cloudflare.log")
tunnel_log_file = tunnel_log.open("w")
mpt_cloudflare_proc = subprocess.Popen(
    [str(binary), "tunnel", "--protocol", "http2",
     "--url", f"http://127.0.0.1:{PORT}", "--allowed-mail", mail],
    stdout=tunnel_log_file, stderr=subprocess.STDOUT
)
del mail

url_pattern = re.compile(r"https://[a-z0-9-]+\\.trycloudflare\\.com")
for _ in range(75):
    time.sleep(2)
    message = tunnel_log.read_text(errors="replace")
    found = url_pattern.search(message)
    if found:
        print("\nLISTO: abre este enlace privado desde tu Samsung:")
        print(found.group(0))
        print("Cloudflare te enviara un codigo a tu correo.")
        print("No compartas el enlace ni tus claves. Colab debe permanecer conectado.")
        break
    if mpt_cloudflare_proc.poll() is not None:
        raise RuntimeError("Cloudflare se cerro. Logs:\n" + message[-2500:])
else:
    raise RuntimeError("No se genero el enlace. Logs:\n" + message[-2500:])
